# Set Library Path

In [1]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

# Load Libraries

In [3]:
library(readxl)
library(ggplot2)
library(viridis)
library(dplyr)
library(glue)
library(tidyr)

# Load Data

In [4]:
# List all files
files <- list.files(path = "09_WGCNA_GO", pattern = "_enrichr_results\\.xlsx$", full.names = TRUE)

# Extract module names from filenames and filter
module_names <- gsub("09_WGCNA_GO/(.*)_enrichr_results\\.xlsx", "\\1", files)

# View the module names
print(module_names)

 [1] "bisque4"         "black"           "blue"            "brown"          
 [5] "brown4"          "cyan"            "darkgreen"       "darkgrey"       
 [9] "darkmagenta"     "darkolivegreen"  "darkorange"      "darkorange2"    
[13] "darkred"         "darkslateblue"   "darkturquoise"   "floralwhite"    
[17] "green"           "greenyellow"     "grey"            "grey60"         
[21] "ivory"           "lightcyan"       "lightcyan1"      "lightgreen"     
[25] "lightsteelblue1" "lightyellow"     "magenta"         "mediumpurple3"  
[29] "midnightblue"    "orange"          "orangered4"      "paleturquoise"  
[33] "pink"            "plum1"           "plum2"           "purple"         
[37] "red"             "royalblue"       "saddlebrown"     "salmon"         
[41] "salmon4"         "sienna3"         "skyblue"         "skyblue3"       
[45] "steelblue"       "tan"             "thistle1"        "thistle2"       
[49] "turquoise"       "violet"          "white"           "yellow"         

# Combined Dot Plot

In [5]:
# List of GO databases
databases <- c("GO_Biological_Process_2025", "GO_Cellular_Component_2025", "GO_Molecular_Function_2025",
               "KEGG_2021_Human", "Reactome_2022", "RNAseq_DiseaseGene_DrugSigs_GEO")

In [6]:
# Create abbreviation mapping
database_abbreviations <- c(
  "GO_Biological_Process_2025" = "GO BP",
  "GO_Cellular_Component_2025" = "GO CC", 
  "GO_Molecular_Function_2025" = "GO MF",
  "KEGG_2021_Human" = "KEGG",
  "Reactome_2022" = "REACTOME",
  "RNAseq_DiseaseGene_DrugSigs_GEO" = "GEO RNAseq"
)

In [7]:
# Initialize an empty dataframe for all GO data
all_data <- data.frame()

In [8]:
# Iterate over each module and read in the corresponding Excel file
for (file in files) {
  for (database in databases) {
    try({
      # Read the data from the Excel file
      df <- read_excel(file, sheet = database)
      
      if (nrow(df) == 0) {
        next  # Skip to the next database if no data
      }
      
      # Select the required columns and add module and database information
      df <- df %>%
        filter(Adjusted.P.value <= 0.05) %>%
        select(Term, Adjusted.P.value, Odds.Ratio) %>%
        mutate(Module = file, Database = database) %>%
        arrange(Adjusted.P.value)
      
      # Append to the dataframe
      all_data <- bind_rows(all_data, df)
      
    }, silent = TRUE)
  }
}

# Check if any data was read
if (nrow(all_data) == 0) {
  stop("No data read from any of the Excel files.")
}

# Display the combined dataframe
head(all_data)

,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,"Defective EXT1 Causes Exostoses 1, TRPS2 And CHDS R-HSA-3656253",0.03121893,72.49818,09_WGCNA_GO/darkolivegreen_enrichr_results.xlsx,Reactome_2022
2,Receptor-type Tyrosine-Protein Phosphatases R-HSA-388844,0.03121893,44.28889,09_WGCNA_GO/darkolivegreen_enrichr_results.xlsx,Reactome_2022
3,Defective B3GALT6 Causes EDSP2 And SEMDJL1 R-HSA-4420332,0.03121893,44.28889,09_WGCNA_GO/darkolivegreen_enrichr_results.xlsx,Reactome_2022
4,Defective B3GAT3 Causes JDSSDHD R-HSA-3560801,0.03121893,44.28889,09_WGCNA_GO/darkolivegreen_enrichr_results.xlsx,Reactome_2022
5,"Defective B4GALT7 Causes EDS, Progeroid Type R-HSA-3560783",0.03121893,44.28889,09_WGCNA_GO/darkolivegreen_enrichr_results.xlsx,Reactome_2022
6,Synaptic Adhesion-Like Molecules R-HSA-8849932,0.03121893,41.95579,09_WGCNA_GO/darkolivegreen_enrichr_results.xlsx,Reactome_2022


In [9]:
# Format comparison name
all_data$Module <- gsub("09_WGCNA_GO/", "", all_data$Module)
all_data$Module <- gsub("_enrichr_results.xlsx", "", all_data$Module)

# View
head(all_data)

,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,"Defective EXT1 Causes Exostoses 1, TRPS2 And CHDS R-HSA-3656253",0.03121893,72.49818,darkolivegreen,Reactome_2022
2,Receptor-type Tyrosine-Protein Phosphatases R-HSA-388844,0.03121893,44.28889,darkolivegreen,Reactome_2022
3,Defective B3GALT6 Causes EDSP2 And SEMDJL1 R-HSA-4420332,0.03121893,44.28889,darkolivegreen,Reactome_2022
4,Defective B3GAT3 Causes JDSSDHD R-HSA-3560801,0.03121893,44.28889,darkolivegreen,Reactome_2022
5,"Defective B4GALT7 Causes EDS, Progeroid Type R-HSA-3560783",0.03121893,44.28889,darkolivegreen,Reactome_2022
6,Synaptic Adhesion-Like Molecules R-HSA-8849932,0.03121893,41.95579,darkolivegreen,Reactome_2022


In [10]:
# Define keywords
keywords <- c("neuro", "axon", "synapse", "receptor", "signal", "transmit", "develop")

# Create a pattern for grepl
pattern <- paste(keywords, collapse = "|")

# Filter all_data
all_data <- all_data[grepl(pattern, all_data$Term, ignore.case = TRUE), ]

# View filtered data
head(all_data)

,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
2,Receptor-type Tyrosine-Protein Phosphatases R-HSA-388844,3.121893e-02,44.288889,darkolivegreen,Reactome_2022
8,Protein-protein Interactions At Synapses R-HSA-6794362,3.152256e-02,14.307083,darkolivegreen,Reactome_2022
10,TNFR1-induced NFkappaB Signaling Pathway R-HSA-5357956,3.354067e-02,33.206667,darkolivegreen,Reactome_2022
12,NOD1/2 Signaling Pathway R-HSA-168638,4.488429e-02,25.699355,darkolivegreen,Reactome_2022
14,Embryonic Skeletal System Development (GO:0048706),6.514028e-04,29.459259,darkorange,GO_Biological_Process_2025
60,Ptbp1 Neuronal progenitor cell (mNPC) knockdown GSE70985 down,7.260122e-08,3.196997,green,RNAseq_DiseaseGene_DrugSigs_GEO


In [14]:
# Generate dot plot
if (nrow(all_data) > 0) {
  
  # Create database abbreviations and assign colors
  all_data <- all_data %>%
    mutate(
      DB_Abbrev = ifelse(Database %in% names(database_abbreviations), 
                         database_abbreviations[Database], Database),
      Module = factor(Module),
      # Create simple term label with database
      Simple_Term = paste(DB_Abbrev, Term, sep = ": ")
    )
  
  # Create a color mapping for databases
  unique_dbs <- unique(all_data$DB_Abbrev)
  db_colors <- setNames(RColorBrewer::brewer.pal(max(3, length(unique_dbs)), "Set3")[1:length(unique_dbs)], 
                       unique_dbs)
  
  # Sort terms by database first, then by term name
  all_data <- all_data %>%
    arrange(DB_Abbrev, Term)
  
  # Ensure terms are ordered properly
  term_order <- unique(all_data$Simple_Term)
  all_data$Simple_Term <- factor(all_data$Simple_Term, levels = term_order)
  
  dot_plot <- ggplot(all_data, aes(x = Module, y = Simple_Term, size = Odds.Ratio, fill = Adjusted.P.value)) +
    geom_point(shape = 21, alpha = 0.8) +
    scale_fill_viridis(name = 'Adjusted P-value', direction = -1) +
    scale_size_continuous(name = 'Odds Ratio', range = c(1, 8)) +
    xlab('') + ylab('') +
    labs(title = 'Significant GO Terms for WGCNA Modules') +
    theme_minimal() +
    theme(
      panel.background = element_rect(fill = "white", color = NA),
      panel.grid.major = element_blank(),
      panel.grid.minor = element_line(color = "black"),
      axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5),
      axis.text.y = element_text(size = 7),
      legend.position = "right",
      plot.title = element_text(size = 11)
    )
  
  # Save the plot
  ggsave(filename = "09_WGCNA_GO/combined_neurological_terms_dot_plot_adjpval.pdf", 
         plot = dot_plot, height = 5, width = 8)
}